# Robco Arm Environment Demo with EGL Rendering

This notebook demonstrates how to load and render the Robco arm environment using EGL backend for headless rendering, perfect for SSH connections.

## Overview
- Configure MuJoCo to use EGL rendering backend
- Load the custom Robco arm environments
- Test both RobcoArm and RobcoReach environments
- Render and visualize the robot in action

In [1]:
# Import Dependencies
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import jax
import jax.numpy as jp

# Set environment variables for EGL rendering BEFORE importing mujoco
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

import mujoco
from mujoco import mjx

# Add the project to Python path
sys.path.append('/home/duckoid/Downloads/mujoco_playground')

# Import our custom robco environments
from mujoco_playground._src import robco

In [2]:
# Load Available Robco Environments
print("Available Robco environments:")
print(robco.ALL_ENVS)

# Get default configuration for RobcoArm
config = robco.get_default_config("RobcoArm")
print(f"\nDefault config for RobcoArm:")
for key, value in config.items():
    print(f"  {key}: {value}")

Available Robco environments:
('RobcoArm',)

Default config for RobcoArm:
  action_repeat: 1
  action_scale: 1.0
  ctrl_dt: 0.02
  episode_length: 1000
  impl: jax
  nconmax: 10
  njmax: 2
  sim_dt: 0.002
  vision: False


In [3]:
# Load RobcoArm Environment
try:
    env = robco.load("RobcoArm")
    print("RobcoArm environment loaded successfully!")
    print(f"Action space size: {env.action_size}")
    print(f"Number of joints: {env.num_joints}")
    print(f"XML path: {env.xml_path}")
        
except Exception as e:
    print(f"Error loading RobcoArm: {e}")
    import traceback
    traceback.print_exc()

RobcoArm environment loaded successfully!
Action space size: 6
Number of joints: 6
XML path: /home/duckoid/Downloads/mujoco_playground/mujoco_playground/_src/robco/xmls/robco_simple.xml


In [4]:
# Test Rendering with EGL
def render_environment(env, width=480, height=480, camera_name="front_camera"):
    """Render the environment using MuJoCo's EGL backend."""
    try:
        # Create a renderer
        renderer = mujoco.Renderer(env.mj_model, height=height, width=width)
        
        # Create fresh data for rendering
        mj_data = mujoco.MjData(env.mj_model)
        
        # Set the robot to a neutral pose
        mj_data.qpos[:] = 0.0  # All joints at zero position
        
        # Forward kinematics to update positions
        mujoco.mj_forward(env.mj_model, mj_data)
        
        # Update the scene with the specified camera
        renderer.update_scene(mj_data, camera=camera_name)
        pixels = renderer.render()
        
        # Close the renderer
        renderer.close()
        
        return pixels
    except Exception as e:
        print(f"Rendering error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Test rendering with different cameras
if 'env' in locals():
    print("Testing EGL rendering...")
    
    # List available cameras
    try:
        print(f"Available cameras in model:")
        for i in range(env.mj_model.ncam):
            camera_name = mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_CAMERA, i)
            print(f"  Camera {i}: {camera_name}")
    except:
        print("Could not list cameras")
    
    # Try rendering with the front camera
    pixels = render_environment(env, camera_name="front_camera")
    if pixels is not None:
        print(f"Successfully rendered frame: {pixels.shape}")
        
        # Display the rendered frame
        plt.figure(figsize=(10, 5))
        
        # Render from front camera
        plt.subplot(1, 2, 1)
        plt.imshow(pixels)
        plt.title("Robco Arm - Front Camera")
        plt.axis('off')
        
        # Render from side camera
        pixels_side = render_environment(env, camera_name="side_camera")
        if pixels_side is not None:
            plt.subplot(1, 2, 2)
            plt.imshow(pixels_side)
            plt.title("Robco Arm - Side Camera")
            plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("Rendering failed")

Testing EGL rendering...
Available cameras in model:
Rendering error: The camera "front_camera" does not exist.
Rendering failed


Traceback (most recent call last):
  File "/tmp/ipykernel_22062/1873647934.py", line 18, in render_environment
    renderer.update_scene(mj_data, camera=camera_name)
  File "/home/duckoid/Downloads/mujoco_playground/.venv/lib/python3.11/site-packages/mujoco/renderer.py", line 277, in update_scene
    raise ValueError(f'The camera "{camera}" does not exist.')
ValueError: The camera "front_camera" does not exist.
